<a href="https://colab.research.google.com/github/shentan-shiina/Colab_Deployment_Log/blob/main/openpi_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 😺 Colab implementation for the [openpi](https://github.com/Physical-Intelligence/openpi) repo (Inference)

## Implementation Details
- We use [uv](https://docs.astral.sh/uv/guides/install-python/) for package and virtual environment managing.
- The default /usr/local/lib/python3.12 is used for the openpi model server: **Default Python3.12 -> $\pi$ model**
- A venv libero /content/openpi/examples/libero/.venv is created for the LIBERO evaluation client: **libero Python3.8 -> LIBERO Benchmark**
- For runing the server and client at the same time, we use multi-threading with a ouput capturing subprocess.

## TODO List
- [x] LIBERO evaluation environment
- [ ] CALVIN evaluation environment
- [ ] LIBERO-PLUS evaluation environment
- [ ] LIBERO-Pro evaluation environment
- [ ] Finetuning with openpi repo data
- [ ] Finetuning with new benchmark data
- [ ] Finetuning with custom data

## Miscellaneous
- We tried [condacolab](https://github.com/conda-incubator/condacolab), [konda](https://github.com/tamnguyenvan/konda), they are too heavy and rely on miniconda. On the other hand, uv supports fast, direct, and parameterized venv install, which solves a big pain. You may refer to our earlier trials for [creating a libero env with condacolab/konda](https://colab.research.google.com/drive/1PnzXfoWBWzjw5WaBJy6IKRtSaPTvirDd)(It also contains some tricks for runing venv in your cell)


## Setup [openpi](https://github.com/Physical-Intelligence/openpi) repo

In [ ]:
!nvidia-smi --query-gpu=gpu_name --format=csv

name
NVIDIA A100-SXM4-80GB


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
from google.colab import auth
auth.authenticate_user()

In [ ]:
!git clone --recurse-submodules https://github.com/Physical-Intelligence/openpi.git

Cloning into 'openpi'...
remote: Enumerating objects: 1561, done.
remote: Total 1561 (delta 0), reused 0 (delta 0), pack-reused 1561 (from 1)
Receiving objects: 100% (1561/1561), 24.61 MiB | 10.55 MiB/s, done.
Resolving deltas: 100% (846/846), done.
Submodule 'third_party/aloha' (https://github.com/Physical-Intelligence/aloha.git) registered for path 'third_party/aloha'
Submodule 'third_party/libero' (https://github.com/Lifelong-Robot-Learning/LIBERO.git) registered for path 'third_party/libero'
Cloning into '/content/openpi/third_party/aloha'...
remote: Enumerating objects: 259, done.        
remote: Counting objects: 100% (91/91), done.        
remote: Compressing objects: 100% (67/67), done.        
remote: Total 259 (delta 43), reused 55 (delta 20), pack-reused 168 (from 1)        
Receiving objects: 100% (259/259), 42.74 MiB | 14.88 MiB/s, done.
Resolving deltas: 100% (120/120), done.
Cloning into '/content/openpi/third_party/libero'...
remote: Enumerating objects: 1788, done.    

### Install uv

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.9.30 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!


### Install openpi env dependencies

In [ ]:
%cd openpi

/content/openpi


In [ ]:
!GIT_LFS_SKIP_SMUDGE=1 uv sync
!GIT_LFS_SKIP_SMUDGE=1 uv pip install -e .

Using CPython 3.11.14
Creating virtual environment at: .venv
Resolved 281 packages in 1ms
Prepared 242 packages in 1m 06s
Installed 242 packages in 323ms
 + absl-py==2.3.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.12.4
 + aiosignal==1.3.2
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + asttokens==3.0.0
 + attrs==25.3.0
 + augmax==0.4.1
 + av==14.4.0
 + beartype==0.19.0
 + beautifulsoup4==4.13.4
 + blinker==1.9.0
 + cachetools==5.5.2
 + certifi==2025.4.26
 + cffi==1.17.1
 + cfgv==3.4.0
 + charset-normalizer==3.4.2
 + chex==0.1.89
 + click==8.2.1
 + cloudpickle==3.1.1
 + cmake==4.0.2
 + comm==0.2.2
 + contourpy==1.3.2
 + crc32c==2.7.1
 + cycler==0.12.1
 + datasets==3.6.0
 + debugpy==1.8.14
 + decorator==5.2.1
 + deepdiff==8.5.0
 + diffusers==0.33.1
 + dill==0.3.8
 + distlib==0.3.9
 + dm-control==1.0.14
 + dm-env==1.6
 + dm-tree==0.1.9
 + docker-pycreds==0.4.0
 + docstring-parser==0.16
 + donfig==0.8.1.post1
 + draccus==0.10.0
 + einops==0.8.1
 + equinox==0.12.2
 + etils==1.1

In [ ]:
%env PYTHONPATH=$PYTHONPATH:/content/openpi/src

env: PYTHONPATH=$PYTHONPATH:/content/openpi/src


## Test for server environment
<u>🥺**Caution, you need to restart the session to make editable packages apply**</u>




In [ ]:
import dataclasses

import jax

from openpi.models import model as _model
from openpi.policies import droid_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

/usr/local/lib/python3.12/dist-packages/ml_collections/config_dict/config_dict.py:163: SyntaxWarning: invalid escape sequence '\['
  index_match = re.match("(.*)\[([0-9]+)\]", key)


In [ ]:
# Change the default data path
%env OPENPI_DATA_HOME=/content/openpi/openpi-assets

env: OPENPI_DATA_HOME=/content/openpi/openpi-assets


In [ ]:
config = _config.get_config("pi0_fast_droid")
checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi0_fast_droid")

# Create a trained policy.
policy = _policy_config.create_trained_policy(config, checkpoint_dir)

# Run inference on a dummy example. This example corresponds to observations produced by the DROID runtime.
example = droid_policy.make_droid_example()
result = policy.infer(example)

# Delete the policy to free up memory.
del policy

print("Actions shape:", result["actions"].shape)

  0%|          | 0.00/10.1G [00:00<?, ?iB/s]

  0%|          | 0.00/4.07M [00:00<?, ?iB/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/253 [00:00<?, ?B/s]

processing_action_tokenizer.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/physical-intelligence/fast:
- processing_action_tokenizer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]

Actions shape: (10, 8)


In [ ]:
# Download the whole datasets here
#!curl -L -o /content/drive/MyDrive/Colab/openpi-assets\
#  https://www.kaggle.com/api/v1/datasets/download/justsahil/openpi-assets

In [ ]:
# print(checkpoint_dir)

In [ ]:
#!cp -r /root/.cache/openpi/openpi-assets/ /content/openpi/

In [ ]:
#!cp -r /content/openpi /content/drive/MyDrive/Colab/2Workspaces/openpi

## Create sub-environment for evaluation

In [ ]:
%cd /content/openpi

/content/openpi


In [ ]:
!uv venv --python 3.8 examples/libero/.venv
!uv pip install \
    --python examples/libero/.venv/bin/python \
    -r examples/libero/requirements.txt \
    -r third_party/libero/requirements.txt \
    --extra-index-url https://download.pytorch.org/whl/cu113 \
    --index-strategy=unsafe-best-match
!uv pip install -e packages/openpi-client\
    --python examples/libero/.venv/bin/python
!uv pip install -e third_party/libero\
    --python examples/libero/.venv/bin/python

Using CPython 3.8.20
Creating virtual environment at: examples/libero/.venv
Activate with: source examples/libero/.venv/bin/activate
Using Python 3.8.20 environment at: examples/libero/.venv
Resolved 121 packages in 6.50s
Prepared 97 packages in 21.13s
Installed 121 packages in 222ms
 + absl-py==2.1.0
 + antlr4-python3-runtime==4.9.3
 + attrs==25.3.0
 + bddl==1.0.1
 + certifi==2024.12.14
 + cffi==1.17.1
 + charset-normalizer==3.4.0
 + click==8.1.8
 + cloudpickle==2.1.0
 + cryptography==45.0.7
 + cycler==0.12.1
 + docker-pycreds==0.4.0
 + docstring-parser==0.16
 + easydict==1.9
 + egl-probe==1.0.2
 + einops==0.4.1
 + etils==1.3.0
 + eval-type-backport==0.2.0
 + evdev==1.7.1
 + exceptiongroup==1.3.1
 + fastjsonschema==2.21.2
 + filelock==3.16.1
 + fonttools==4.55.3
 + fsspec==2025.3.0
 + future==0.18.2
 + gitdb==4.0.12
 + gitpython==3.1.46
 + glfw==1.12.0
 + google-auth==2.48.0
 + google-auth-oauthlib==1.0.0
 + grpcio==1.70.0
 + gym==0.25.2
 + gym-notices==0.1.0
 + h5py==3.11.0
 + hf-xet

In [ ]:
%env PYTHONPATH=$PYTHONPATH:/content/openpi/third_party/libero

env: PYTHONPATH=$PYTHONPATH:/content/openpi/third_party/libero


## 🚀 Run evaluation client and model server scripts

In [ ]:
# The client and server scripts
!examples/libero/.venv/bin/python examples/libero/main.py # run this beforehand to cache for user input
#!uv run scripts/serve_policy.py --env LIBERO

Do you want to specify a custom path for the dataset folder? (Y/N): n
Initializing the default config file...
The following information is stored in the config file: /root/.libero/config.yaml
benchmark_root: /content/openpi/third_party/libero/libero/libero
bddl_files: /content/openpi/third_party/libero/libero/libero/./bddl_files
init_states: /content/openpi/third_party/libero/libero/libero/./init_files
datasets: /content/openpi/third_party/libero/libero/libero/../datasets
assets: /content/openpi/third_party/libero/libero/libero/./assets
[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /content/openpi/examples/libero/.venv/lib/python3.8/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-

KeyboardInterrupt: 

In [ ]:
import subprocess
import threading
import sys
import time
# Edit the scripts here
server_cmd = "uv run scripts/serve_policy.py --env LIBERO"
client_cmd = "examples/libero/.venv/bin/python examples/libero/main.py"

def stream_output(process, label):
    """Captures and prints output line-by-line with a label."""
    for line in iter(process.stdout.readline, ''):
        if line:
            print(f"[{label}] {line.strip()}")
    process.stdout.close()

# 1. Start the Server Process

server_proc = subprocess.Popen(
    server_cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 2. Start the Client Process (with a short delay)
time.sleep(1) # Adjust based on how long your server takes to load

client_proc = subprocess.Popen(
    client_cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 3. Create threads to monitor both outputs simultaneously
t1 = threading.Thread(target=stream_output, args=(server_proc, "SERVER"))
t2 = threading.Thread(target=stream_output, args=(client_proc, "CLIENT"))

t1.start()
t2.start()

try:
    # Keep the cell running while processes are active
    while server_proc.poll() is None or client_proc.poll() is None:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n--- Stopping processes... ---")
    server_proc.terminate()
    client_proc.terminate()

[SERVER] warning: The `tool.uv.dev-dependencies` field (used in `packages/openpi-client/pyproject.toml`) is deprecated and will be removed in a future release; use `dependency-groups.dev` instead
[CLIENT] [robosuite WARNING] No private macro file found! (macros.py:53)
[CLIENT] [robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[CLIENT] [robosuite WARNING] To setup, run: python /content/openpi/examples/libero/.venv/lib/python3.8/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)
[CLIENT] Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
[CLIENT] Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
[CLIENT] See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
[CLIENT] INFO:root:Task suite: libero_spatial
[CLIENT] INFO:root:Waiting for server a

### You may need to set environment again after restarting session

In [ ]:
%cd /content/openpi
%env OPENPI_DATA_HOME=/content/openpi/openpi-assets
%env PYTHONPATH=$PYTHONPATH:/content/openpi/third_party/libero

Thu Feb  5 09:04:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             58W /  400W |   61377MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----